# 🧠 SMORE: MRI Super-Resolution & Anti-Aliasing
### Digital Image Processing — Advanced Project
**Team:** Muhammad Kashif (231568) | Abdulrehman Afaq (231577) | Umair Habib (231694)

**Paper:** SMORE: A Self-Supervised Anti-Aliasing and Super-Resolution Algorithm for MRI Using Deep Learning  
**Dataset:** IXI Brain MRI (via Kaggle — preprocessed by CAT12/SPM)

---
## 📋 Notebook Outline
1. Install & Import Libraries
2. Download Dataset from Kaggle
3. Explore & Load Real NIfTI Files
4. Extract 2D Slices from 3D Volumes
5. Visualize Real MRI Slices
6. Simulate Low-Resolution & Aliasing
7. Build Custom Dataset & DataLoader
8. Define U-Net Model (SMORE Backbone)
9. Train the Model
10. Evaluate: PSNR, SSIM & Visual Comparison
11. Ablation Study (Bicubic vs CNN vs Full Pipeline)
12. Final Results Summary

> ⚠️ **Before running:** Make sure your Colab runtime is set to **GPU** (Runtime → Change runtime type → T4 GPU)

---
## Step 1 — Install & Import Libraries

In [ ]:
# Install required libraries
%pip install nibabel scikit-image tqdm kagglehub -q
print("✅ Libraries installed successfully")

In [ ]:
import os
import glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from skimage import transform as skimage_transform
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ All libraries imported")
print(f"🖥️  Using device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   ⚠️ No GPU found. Training will be slow. Go to Runtime → Change runtime type → T4 GPU")

# Repository root (for src/smore imports in local clone or Colab)
import sys
from pathlib import Path
_repo_root = Path.cwd()
if not (_repo_root / "src").is_dir():
    _repo_root = _repo_root.parent
if (_repo_root / "src").is_dir() and str(_repo_root / "src") not in sys.path:
    sys.path.insert(0, str(_repo_root / "src"))


---
## Step 2 — Download Dataset from Kaggle

We use the preprocessed IXI dataset from Kaggle. It contains skull-stripped, MNI-registered brain MRI files in NIfTI format, processed using CAT12/SPM.

> ⏳ **This will download ~4.2 GB** — takes about 2–3 minutes on Colab. Run once and it caches automatically.

In [ ]:
import kagglehub

print("⬇️ Downloading dataset... (this takes 2-3 minutes, please wait)")
DATASET_PATH = kagglehub.dataset_download("hamedamin/preprocessed-oasis-and-epilepsy-and-ixi")
print(f"✅ Dataset downloaded to: {DATASET_PATH}")

---
## Step 3 — Explore & Load Real NIfTI Files

The dataset has two main folders:
- `mr_IXI_480i/` → Gray matter (`mwp1*.nii`) and White matter (`mwp2*.nii`) segmentation maps
- `mri_IXI_480_2/` → **Skull-stripped registered full-brain MRI (`wm*.nii`)** ← We use this
- `mri_IXI_480_2/` → Jacobian maps (`wj*.nii`) — not needed for SR

We use the `wm*.nii` files because they contain full brain structure — ideal for super-resolution.

In [ ]:
# Find all wm*.nii files (skull-stripped registered MRI)
WM_FOLDER = os.path.join(DATASET_PATH, "mri_IXI_480_2")
all_nii_files = sorted(glob.glob(os.path.join(WM_FOLDER, "wm*.nii")))

# Filter out corrupt/empty files (some are 0 MB in the dataset)
valid_nii_files = [f for f in all_nii_files if os.path.getsize(f) > 500_000]  # > 0.5 MB

print(f"📁 Total wm*.nii files found : {len(all_nii_files)}")
print(f"✅ Valid files (non-empty)   : {len(valid_nii_files)}")
print(f"\nExample files:")
for f in valid_nii_files[:5]:
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {size_mb:.2f} MB  →  {os.path.basename(f)}")

In [ ]:
# Inspect one NIfTI volume to understand shape and values
sample_nii = nib.load(valid_nii_files[0])
sample_vol  = sample_nii.get_fdata()

print("=" * 50)
print("📊 Sample NIfTI File Info")
print("=" * 50)
print(f"File     : {os.path.basename(valid_nii_files[0])}")
print(f"Shape    : {sample_vol.shape}   (H × W × D slices)")
print(f"Dtype    : {sample_vol.dtype}")
print(f"Min val  : {sample_vol.min():.4f}")
print(f"Max val  : {sample_vol.max():.4f}")
print(f"Mean val : {sample_vol.mean():.4f}")
print(f"Voxel spacing (mm): {sample_nii.header.get_zooms()}")
print("=" * 50)

---
## Step 4 — Extract 2D Slices from 3D Volumes

MRI volumes are 3D (H × W × D). We extract individual 2D axial slices for training.  
We skip near-empty slices (pure background, top/bottom of skull) to keep only informative brain slices.

> ⚙️ **Config:** Using 30 subjects and fixed 128×128 slice size for Colab compatibility.

In [ ]:
# ============================================================
# Configuration — adjust these if needed
# ============================================================
NUM_SUBJECTS    = 30       # Number of MRI volumes to use (increase if you have more RAM)
SLICE_SIZE      = (128, 128)  # All slices resized to this (must be power of 2 for U-Net)
DOWNSAMPLE_FACTOR = 4      # SR factor: 4x means LR is 1/4 resolution of HR
EMPTY_THRESHOLD  = 0.02    # Skip slices where max intensity is below this (background)
# ============================================================

def extract_slices(nii_path, target_size=SLICE_SIZE, axis=2, empty_thresh=EMPTY_THRESHOLD):
    """
    Load a 3D NIfTI volume and extract non-empty 2D axial slices.
    Each slice is resized to target_size.
    Returns a list of 2D float32 numpy arrays, normalized to [0, 1].
    """
    vol = nib.load(nii_path).get_fdata().astype(np.float32)

    # Global normalization of the entire volume to [0, 1]
    vol_min, vol_max = vol.min(), vol.max()
    if vol_max - vol_min < 1e-6:
        return []   # Skip completely empty volumes
    vol = (vol - vol_min) / (vol_max - vol_min)

    slices = []
    n_slices = vol.shape[axis]
    for i in range(n_slices):
        sl = vol[:, :, i] if axis == 2 else vol[:, i, :] if axis == 1 else vol[i, :, :]

        # Skip empty/background slices
        if sl.max() < empty_thresh:
            continue

        # Resize to fixed target size
        sl_resized = skimage_transform.resize(
            sl, target_size,
            anti_aliasing=True,
            preserve_range=True
        ).astype(np.float32)

        slices.append(sl_resized)

    return slices


# Extract slices from all selected subjects
print(f"📂 Extracting slices from {NUM_SUBJECTS} subjects...")
slices_by_subject = []
selected_files = valid_nii_files[:NUM_SUBJECTS]

for fpath in tqdm(selected_files, desc="Loading volumes"):
    slices = extract_slices(fpath)
    if slices:
        slices_by_subject.append(slices)

all_slices = [sl for subject in slices_by_subject for sl in subject]

print(f"\n✅ Total 2D slices extracted : {len(all_slices)}")
print(f"   Subjects with valid slices : {len(slices_by_subject)}")
print(f"   Slice shape               : {all_slices[0].shape}")
print(f"   Value range               : [{min(s.min() for s in all_slices[:10]):.3f}, {max(s.max() for s in all_slices[:10]):.3f}]")


---
## Step 5 — Visualize Real MRI Slices

Let's look at some actual brain MRI slices from our dataset before training.

In [ ]:
# Show a grid of 12 real MRI slices from the dataset
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
fig.suptitle('Real IXI Brain MRI Slices (wm.nii — Skull-Stripped, MNI-Registered)',
             fontsize=13, fontweight='bold', y=1.01)

# Pick evenly spaced slices across the dataset
indices = np.linspace(0, len(all_slices) - 1, 12, dtype=int)

for ax, idx in zip(axes.flat, indices):
    ax.imshow(all_slices[idx], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Slice #{idx}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('real_mri_slices.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Real MRI slices visualized")

---
## Step 6 — Simulate Low-Resolution & Aliasing

SMORE is a **self-supervised** method — it learns from the image itself, not from external HR/LR pairs.  
We simulate the degradation pipeline:

1. **Bicubic LR**: Downsample by `DOWNSAMPLE_FACTOR` with anti-aliasing, then bicubic upsample back → smooth but blurry input  
2. **Aliased LR**: Downsample with nearest-neighbor (no anti-aliasing) → introduces ringing/aliasing artifacts

In [ ]:
def simulate_lr_aliased(hr_slice, downsample_factor=DOWNSAMPLE_FACTOR):
    """
    Simulate low-resolution + aliasing artifacts.
    Input : hr_slice — 2D numpy array, normalized [0,1]
    Output: lr_aliased — same shape as input but with aliasing artifacts
    """
    H, W = hr_slice.shape
    lr_H, lr_W = H // downsample_factor, W // downsample_factor

    # Step 1: Downsample WITHOUT anti-aliasing (nearest-neighbor → aliasing)
    lr_aliased = skimage_transform.resize(
        hr_slice, (lr_H, lr_W),
        order=0,               # Nearest-neighbor
        anti_aliasing=False,
        preserve_range=True
    )
    # Step 2: Upsample back to original size with bicubic
    lr_aliased_up = skimage_transform.resize(
        lr_aliased, (H, W),
        order=3,               # Bicubic
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

    return np.clip(lr_aliased_up, 0, 1)


def simulate_lr_bicubic(hr_slice, downsample_factor=DOWNSAMPLE_FACTOR):
    """
    Simulate smooth low-resolution (bicubic baseline — no aliasing).
    """
    H, W = hr_slice.shape
    lr_H, lr_W = H // downsample_factor, W // downsample_factor

    # Downsample WITH anti-aliasing
    lr_clean = skimage_transform.resize(
        hr_slice, (lr_H, lr_W),
        anti_aliasing=True,
        preserve_range=True
    )
    # Upsample with bicubic
    lr_bicubic_up = skimage_transform.resize(
        lr_clean, (H, W),
        order=3,
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

    return np.clip(lr_bicubic_up, 0, 1)


# Visualize the degradation on 3 sample slices
sample_indices = [len(all_slices)//4, len(all_slices)//2, 3*len(all_slices)//4]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle(f'Degradation Simulation (Downsample Factor = {DOWNSAMPLE_FACTOR}x)',
             fontsize=13, fontweight='bold')

col_titles = ['Original HR (Ground Truth)', 'Bicubic LR (Blurry)',
              'Aliased LR (Artifacts)', 'Diff: HR - Bicubic']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=9, fontweight='bold')

for row, idx in enumerate(sample_indices):
    hr  = all_slices[idx]
    lrb = simulate_lr_bicubic(hr)
    lra = simulate_lr_aliased(hr)
    diff = np.abs(hr - lrb)

    for col, (img, cmap, vmin, vmax) in enumerate([
        (hr,   'gray',    0, 1),
        (lrb,  'gray',    0, 1),
        (lra,  'gray',    0, 1),
        (diff, 'hot',     0, diff.max())
    ]):
        axes[row][col].imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
        axes[row][col].axis('off')

    # Add PSNR annotations
    psnr_b = peak_signal_noise_ratio(hr, lrb, data_range=1.0)
    psnr_a = peak_signal_noise_ratio(hr, lra, data_range=1.0)
    axes[row][1].set_xlabel(f'PSNR = {psnr_b:.2f} dB', fontsize=8)
    axes[row][2].set_xlabel(f'PSNR = {psnr_a:.2f} dB', fontsize=8)

plt.tight_layout()
plt.savefig('degradation_simulation.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Degradation simulation visualized")

---
## Step 7 — Custom Dataset & DataLoader

We build a PyTorch `Dataset` that:
- Takes a pre-extracted list of HR slices
- Simulates LR+aliased input on-the-fly during training
- Returns `(input_tensor, target_tensor)` pairs for the model

In [ ]:
class MRISliceDataset(Dataset):
    """
    PyTorch Dataset for MRI Super-Resolution.
    - Input  (X): Aliased low-resolution slice (simulated)
    - Target (Y): Original high-resolution slice (ground truth)
    """
    def __init__(self, slices_list, downsample_factor=DOWNSAMPLE_FACTOR):
        self.slices = slices_list
        self.ds_factor = downsample_factor

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        hr = self.slices[idx]   # Already float32, shape (H, W), values [0, 1]

        # Simulate degraded input (aliased LR)
        lr_input = simulate_lr_aliased(hr, self.ds_factor)

        # Convert to tensors: shape (1, H, W)  — 1 channel (grayscale)
        hr_tensor = torch.from_numpy(hr).unsqueeze(0)       # Target
        lr_tensor = torch.from_numpy(lr_input).unsqueeze(0) # Input

        return lr_tensor, hr_tensor


# ── Subject-level train / val / test split (no patient leakage) ──
from smore.splits import subject_level_split

split = subject_level_split(slices_by_subject, train_ratio=0.70, val_ratio=0.15, seed=42)
train_slices = split["train_slices"]
val_slices   = split["val_slices"]
test_slices  = split["test_slices"]

print("✅ Subject-level dataset split (no overlap between partitions):")
print(f"   Train : {split['train_subjects']} subjects | {len(train_slices):,} slices")
print(f"   Val   : {split['val_subjects']} subjects | {len(val_slices):,} slices")
print(f"   Test  : {split['test_subjects']} subjects | {len(test_slices):,} slices")

# ── DataLoaders ─────────────────────────────────────────────
BATCH_SIZE = 16  # Reduce to 8 if you get CUDA out-of-memory errors

train_dataset = MRISliceDataset(train_slices)
val_dataset   = MRISliceDataset(val_slices)
test_dataset  = MRISliceDataset(test_slices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\n✅ DataLoaders created")
print(f"   Batches per epoch (train): {len(train_loader)}")

# Quick sanity check on one batch
inputs, targets = next(iter(train_loader))
print(f"\n🔍 Batch check:")
print(f"   inputs  shape : {inputs.shape}   dtype: {inputs.dtype}")
print(f"   targets shape : {targets.shape}  dtype: {targets.dtype}")
print(f"   input  range  : [{inputs.min():.3f}, {inputs.max():.3f}]")
print(f"   target range  : [{targets.min():.3f}, {targets.max():.3f}]")


---
## Step 8 — Define the U-Net Model (SMORE Backbone)

We implement a **U-Net** with skip connections — the standard backbone used in SMORE and most medical image super-resolution work.

Architecture:
- **Encoder**: 4 downsampling blocks (Conv → BN → ReLU → MaxPool)
- **Bottleneck**: Deepest feature representation
- **Decoder**: 4 upsampling blocks with skip connections from encoder
- **Output**: 1-channel grayscale image (same size as input)

> ✅ **Bug Fix Applied**: Skip connections use `F.interpolate` to handle odd spatial dimensions, preventing the `RuntimeError: Sizes of tensors must match` error from your original notebook.

In [ ]:
class ConvBlock(nn.Module):
    """Two consecutive Conv2d → BatchNorm → ReLU layers."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """
    U-Net for MRI Super-Resolution.
    Input  : (B, 1, H, W)  — aliased low-res slice
    Output : (B, 1, H, W)  — reconstructed high-res slice
    """
    def __init__(self, in_channels=1, out_channels=1, base_features=32):
        super().__init__()
        f = base_features  # shorthand

        # ── Encoder ──────────────────────────────────────────
        self.enc1 = ConvBlock(in_channels, f)       # → f  channels
        self.enc2 = ConvBlock(f,  f * 2)            # → 2f channels
        self.enc3 = ConvBlock(f * 2, f * 4)         # → 4f channels
        self.enc4 = ConvBlock(f * 4, f * 8)         # → 8f channels
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # ── Bottleneck ───────────────────────────────────────
        self.bottleneck = ConvBlock(f * 8, f * 16)  # → 16f channels

        # ── Decoder ──────────────────────────────────────────
        self.up4   = nn.ConvTranspose2d(f * 16, f * 8,  kernel_size=2, stride=2)
        self.dec4  = ConvBlock(f * 16, f * 8)

        self.up3   = nn.ConvTranspose2d(f * 8,  f * 4,  kernel_size=2, stride=2)
        self.dec3  = ConvBlock(f * 8,  f * 4)

        self.up2   = nn.ConvTranspose2d(f * 4,  f * 2,  kernel_size=2, stride=2)
        self.dec2  = ConvBlock(f * 4,  f * 2)

        self.up1   = nn.ConvTranspose2d(f * 2,  f,      kernel_size=2, stride=2)
        self.dec1  = ConvBlock(f * 2,  f)

        # ── Output ───────────────────────────────────────────
        self.out_conv = nn.Conv2d(f, out_channels, kernel_size=1)
        self.sigmoid  = nn.Sigmoid()   # Output in [0, 1]

    def _match_and_cat(self, upsampled, skip):
        """
        Resize 'upsampled' to match 'skip' spatial dims, then concatenate.
        This fixes the odd-dimension size mismatch in skip connections.
        """
        if upsampled.shape[2:] != skip.shape[2:]:
            upsampled = F.interpolate(upsampled, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return torch.cat([upsampled, skip], dim=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b  = self.bottleneck(self.pool(e4))

        # Decoder with skip connections
        d4 = self.dec4(self._match_and_cat(self.up4(b),  e4))
        d3 = self.dec3(self._match_and_cat(self.up3(d4), e3))
        d2 = self.dec2(self._match_and_cat(self.up2(d3), e2))
        d1 = self.dec1(self._match_and_cat(self.up1(d2), e1))

        return self.sigmoid(self.out_conv(d1))


# ── Initialize model ───────────────────────────────────────
model = UNet(in_channels=1, out_channels=1, base_features=32).to(device)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✅ U-Net model created")
print(f"   Total parameters     : {total_params:,}")
print(f"   Trainable parameters : {trainable_params:,}")

# Verify forward pass with a dummy batch
with torch.no_grad():
    dummy_in  = torch.randn(2, 1, SLICE_SIZE[0], SLICE_SIZE[1]).to(device)
    dummy_out = model(dummy_in)
    print(f"   Forward pass OK  : in {dummy_in.shape} → out {dummy_out.shape}")
    print(f"   Output range     : [{dummy_out.min():.3f}, {dummy_out.max():.3f}]")

---
## Step 9 — Train the Model

Training setup:
- **Loss**: Combined **MSE + SSIM** (`CombinedLoss`, weights 1.0 / 0.2)
- **Optimizer**: Adam with learning rate 1e-3
- **Scheduler**: ReduceLROnPlateau — halves LR when val loss plateaus
- **Early Stopping**: Stops training if validation loss doesn't improve for 5 epochs

> ⏱️ **Expected training time on T4 GPU**: ~2–4 minutes for 20 epochs with 30 subjects


In [ ]:
# ── Hyperparameters ────────────────────────────────────────
NUM_EPOCHS      = 20       # Increase to 50+ for better results
LEARNING_RATE   = 1e-3
PATIENCE        = 5        # Early stopping patience
MSE_WEIGHT      = 1.0
SSIM_WEIGHT     = 0.2      # Structural similarity term (v2)

# ── Loss, Optimizer, Scheduler ────────────────────────────
from smore.losses import CombinedLoss

criterion = CombinedLoss(mse_weight=MSE_WEIGHT, ssim_weight=SSIM_WEIGHT)
optimizer  = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# ── Training history ──────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'val_psnr': [], 'val_ssim': []}

best_val_loss  = float('inf')
patience_count = 0
best_model_wts = None


def evaluate(model, loader):
    """Run model on loader, return avg loss, PSNR, SSIM."""
    model.eval()
    total_loss, total_psnr, total_ssim, n = 0, 0, 0, 0

    with torch.no_grad():
        for inputs, targets in loader:
            inputs  = inputs.to(device)
            targets = targets.to(device)
            outputs = model(inputs)

            loss = criterion(outputs, targets)
            total_loss += loss.item() * inputs.size(0)

            # Per-image PSNR and SSIM
            for i in range(inputs.size(0)):
                pred = outputs[i, 0].cpu().numpy()
                tgt  = targets[i, 0].cpu().numpy()
                total_psnr += peak_signal_noise_ratio(tgt, pred, data_range=1.0)
                total_ssim += structural_similarity(tgt, pred, data_range=1.0)
                n += 1

    return total_loss / n, total_psnr / n, total_ssim / n


# ── Training Loop ─────────────────────────────────────────
print(f"🚀 Starting training for {NUM_EPOCHS} epochs...")
print(f"   Batch size    : {BATCH_SIZE}")
print(f"   Learning rate : {LEARNING_RATE}")
print(f"   Device        : {device}")
print("-" * 65)
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>9} | {'Val PSNR':>9} | {'Val SSIM':>9}")
print("-" * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train phase ──────────────────────────────────────
    model.train()
    running_loss = 0.0

    for inputs, targets in train_loader:
        inputs  = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_dataset)

    # ── Validation phase ─────────────────────────────────
    val_loss, val_psnr, val_ssim = evaluate(model, val_loader)
    scheduler.step(val_loss)

    # Store history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_psnr'].append(val_psnr)
    history['val_ssim'].append(val_ssim)

    print(f"{epoch:>6} | {train_loss:>10.6f} | {val_loss:>9.6f} | {val_psnr:>9.4f} | {val_ssim:>9.4f}")

    # ── Early stopping & save best model ─────────────────
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        patience_count = 0
        best_model_wts = {k: v.clone() for k, v in model.state_dict().items()}
        torch.save(best_model_wts, 'best_smore_model.pth')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"\n⏹️  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
            break

print("-" * 65)
print(f"✅ Training complete!")
print(f"   Best val loss : {best_val_loss:.6f}")
print(f"   Best val PSNR : {max(history['val_psnr']):.4f} dB")
print(f"   Best val SSIM : {max(history['val_ssim']):.4f}")

# Load best weights
model.load_state_dict(best_model_wts)
print("✅ Best model weights loaded")


In [ ]:
# ── Plot Training Curves ───────────────────────────────────
epochs_ran = len(history['train_loss'])
x = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Training History', fontsize=13, fontweight='bold')

# Loss
axes[0].plot(x, history['train_loss'], 'b-o', markersize=3, label='Train Loss')
axes[0].plot(x, history['val_loss'],   'r-o', markersize=3, label='Val Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# PSNR
axes[1].plot(x, history['val_psnr'], 'g-o', markersize=3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('PSNR (dB)')
axes[1].set_title('Validation PSNR'); axes[1].grid(True, alpha=0.3)

# SSIM
axes[2].plot(x, history['val_ssim'], 'm-o', markersize=3)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('SSIM')
axes[2].set_title('Validation SSIM'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training curves saved")

---
## Step 10 — Evaluate on Test Set

We now measure final performance on the held-out **test set** using PSNR and SSIM.

In [ ]:
# ── Final evaluation on test set ──────────────────────────
print("📊 Evaluating on test set...")
test_loss, test_psnr, test_ssim = evaluate(model, test_loader)

# Also compute bicubic baseline metrics on the test set
bicubic_psnr_list, bicubic_ssim_list = [], []

for inputs, targets in test_loader:
    for i in range(inputs.size(0)):
        inp = inputs[i, 0].numpy()
        tgt = targets[i, 0].numpy()
        bicubic_psnr_list.append(peak_signal_noise_ratio(tgt, inp, data_range=1.0))
        bicubic_ssim_list.append(structural_similarity(tgt, inp, data_range=1.0))

bicubic_psnr = np.mean(bicubic_psnr_list)
bicubic_ssim = np.mean(bicubic_ssim_list)

# ── Print results table ───────────────────────────────────
print("\n" + "=" * 55)
print("  FINAL TEST SET RESULTS")
print("=" * 55)
print(f"{'Method':<30} {'PSNR (dB)':>10} {'SSIM':>10}")
print("-" * 55)
print(f"{'Bicubic Interpolation (Baseline)':<30} {bicubic_psnr:>10.4f} {bicubic_ssim:>10.4f}")
print(f"{'U-Net (Our Model)':<30} {test_psnr:>10.4f} {test_ssim:>10.4f}")
print("-" * 55)
print(f"{'Improvement (↑ better)':<30} {test_psnr-bicubic_psnr:>+10.4f} {test_ssim-bicubic_ssim:>+10.4f}")
print("=" * 55)

In [ ]:
# ── Visual comparison on 4 test samples ───────────────────
model.eval()

# Grab one batch from the test loader
test_inputs, test_targets = next(iter(test_loader))

with torch.no_grad():
    test_outputs = model(test_inputs.to(device)).cpu()

NUM_SHOW = 4
fig, axes = plt.subplots(NUM_SHOW, 4, figsize=(16, 4 * NUM_SHOW))
fig.suptitle('Visual Comparison on Test Set', fontsize=13, fontweight='bold')

col_headers = ['Input (Aliased LR)', 'Ground Truth (HR)', 'Our Model Output', 'Residual (|GT - Model|)']
for j, title in enumerate(col_headers):
    axes[0, j].set_title(title, fontsize=9, fontweight='bold')

for row in range(NUM_SHOW):
    inp   = test_inputs[row, 0].numpy()
    gt    = test_targets[row, 0].numpy()
    pred  = test_outputs[row, 0].numpy()
    resid = np.abs(gt - pred)

    psnr_in   = peak_signal_noise_ratio(gt, inp,  data_range=1.0)
    psnr_pred = peak_signal_noise_ratio(gt, pred, data_range=1.0)
    ssim_in   = structural_similarity(gt, inp,  data_range=1.0)
    ssim_pred = structural_similarity(gt, pred, data_range=1.0)

    for col, (img, cmap, vmin, vmax) in enumerate([
        (inp,   'gray', 0, 1),
        (gt,    'gray', 0, 1),
        (pred,  'gray', 0, 1),
        (resid, 'hot',  0, resid.max() + 1e-6)
    ]):
        im = axes[row, col].imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_xlabel(f'PSNR={psnr_in:.2f} dB | SSIM={ssim_in:.3f}', fontsize=7)
        elif col == 2:
            axes[row, col].set_xlabel(f'PSNR={psnr_pred:.2f} dB | SSIM={ssim_pred:.3f}', fontsize=7)

plt.tight_layout()
plt.savefig('visual_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Visual comparison saved")

---
## Step 11 — Ablation Study

We compare three approaches on the same test samples:
1. **Bicubic Interpolation** — classical baseline (no learning)
2. **Input (Aliased LR)** — raw degraded input without any correction
3. **Our U-Net** — the SMORE-inspired deep learning model

This validates our model's contribution over classical methods.

In [ ]:
# ── Ablation: collect metrics for all 3 methods on test set ─
ablation = {
    'Aliased LR Input':       {'psnr': [], 'ssim': []},
    'Bicubic Interpolation':  {'psnr': [], 'ssim': []},
    'U-Net (Our Model)':      {'psnr': [], 'ssim': []},
}

model.eval()

for inputs, targets in tqdm(test_loader, desc="Ablation evaluation"):
    with torch.no_grad():
        outputs = model(inputs.to(device)).cpu()

    for i in range(inputs.size(0)):
        inp  = inputs[i, 0].numpy()
        gt   = targets[i, 0].numpy()
        pred = outputs[i, 0].numpy()

        # Bicubic: upsample the LR (already done by simulate_lr_bicubic in dataset)
        # Here we recompute it for clarity
        bicubic = simulate_lr_bicubic(gt)

        for name, img in [
            ('Aliased LR Input',       inp),
            ('Bicubic Interpolation',  bicubic),
            ('U-Net (Our Model)',       pred),
        ]:
            ablation[name]['psnr'].append(peak_signal_noise_ratio(gt, img, data_range=1.0))
            ablation[name]['ssim'].append(structural_similarity(gt, img,   data_range=1.0))

# ── Print ablation table ───────────────────────────────────
print("\n" + "=" * 55)
print("  ABLATION STUDY RESULTS")
print("=" * 55)
print(f"{'Method':<30} {'PSNR (dB)':>10} {'SSIM':>10}")
print("-" * 55)
for name, metrics in ablation.items():
    avg_psnr = np.mean(metrics['psnr'])
    avg_ssim = np.mean(metrics['ssim'])
    print(f"{name:<30} {avg_psnr:>10.4f} {avg_ssim:>10.4f}")
print("=" * 55)

In [ ]:
# ── Bar chart: Ablation Study ─────────────────────────────
methods = list(ablation.keys())
psnr_vals = [np.mean(ablation[m]['psnr']) for m in methods]
ssim_vals = [np.mean(ablation[m]['ssim']) for m in methods]

colors = ['#e74c3c', '#f39c12', '#27ae60']  # Red, Orange, Green

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Ablation Study: PSNR & SSIM Comparison', fontsize=13, fontweight='bold')

bars1 = ax1.bar(methods, psnr_vals, color=colors, edgecolor='black', linewidth=0.8)
ax1.set_ylabel('PSNR (dB)', fontsize=11)
ax1.set_title('Peak Signal-to-Noise Ratio (↑ Higher is Better)')
ax1.set_ylim(min(psnr_vals) * 0.95, max(psnr_vals) * 1.05)
for bar, val in zip(bars1, psnr_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax1.tick_params(axis='x', labelsize=9)
ax1.grid(axis='y', alpha=0.3)

bars2 = ax2.bar(methods, ssim_vals, color=colors, edgecolor='black', linewidth=0.8)
ax2.set_ylabel('SSIM', fontsize=11)
ax2.set_title('Structural Similarity Index (↑ Higher is Better)')
ax2.set_ylim(min(ssim_vals) * 0.95, max(ssim_vals) * 1.05)
for bar, val in zip(bars2, ssim_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax2.tick_params(axis='x', labelsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Ablation study chart saved")

---
## Step 12 — Edge Sharpness Analysis

A key goal of SMORE is to preserve **anatomical edge sharpness**. We use Sobel edge detection to compare edge strength between the input, bicubic baseline, and our model output.

In [ ]:
from skimage.filters import sobel

# Pick 3 test samples for edge analysis
edge_inputs, edge_targets = next(iter(test_loader))

with torch.no_grad():
    edge_outputs = model(edge_inputs.to(device)).cpu()

NUM_EDGE = 3
fig, axes = plt.subplots(NUM_EDGE, 4, figsize=(16, 4 * NUM_EDGE))
fig.suptitle('Edge Sharpness Analysis (Sobel Filter)', fontsize=13, fontweight='bold')

for j, title in enumerate(['Ground Truth', 'Edges: GT', 'Edges: Input (LR)', 'Edges: Our Model']):
    axes[0, j].set_title(title, fontsize=9, fontweight='bold')

for row in range(NUM_EDGE):
    gt   = edge_targets[row, 0].numpy()
    inp  = edge_inputs[row, 0].numpy()
    pred = edge_outputs[row, 0].numpy()

    edges_gt   = sobel(gt)
    edges_inp  = sobel(inp)
    edges_pred = sobel(pred)

    vmax = edges_gt.max()

    axes[row, 0].imshow(gt,         cmap='gray',   vmin=0, vmax=1)
    axes[row, 1].imshow(edges_gt,   cmap='plasma', vmin=0, vmax=vmax)
    axes[row, 2].imshow(edges_inp,  cmap='plasma', vmin=0, vmax=vmax)
    axes[row, 3].imshow(edges_pred, cmap='plasma', vmin=0, vmax=vmax)

    # Edge strength score (mean edge magnitude)
    score_gt   = edges_gt.mean()
    score_inp  = edges_inp.mean()
    score_pred = edges_pred.mean()

    axes[row, 1].set_xlabel(f'Score: {score_gt:.5f}',   fontsize=8)
    axes[row, 2].set_xlabel(f'Score: {score_inp:.5f}',  fontsize=8)
    axes[row, 3].set_xlabel(f'Score: {score_pred:.5f}', fontsize=8)

    for ax in axes[row]:
        ax.axis('off')

plt.tight_layout()
plt.savefig('edge_sharpness.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Edge sharpness analysis saved")

---
## Step 13 — Final Results Summary

A consolidated summary of all results for your project report.

In [ ]:
# ── Final Summary Dashboard ───────────────────────────────
print("\n" + "#" * 60)
print("  SMORE DIP PROJECT — FINAL RESULTS SUMMARY")
print("#" * 60)

print(f"""
📁 DATASET
   Source   : IXI Brain MRI (Kaggle — CAT12/SPM preprocessed)
   Files    : wm*.nii (skull-stripped, MNI-registered)
   Subjects : {NUM_SUBJECTS} ({split['train_subjects']} train / {split['val_subjects']} val / {split['test_subjects']} test)
   Slices   : {len(all_slices):,} total  |  Train: {len(train_slices):,}  Val: {len(val_slices):,}  Test: {len(test_slices):,}
   Split    : Subject-level (no patient leakage)
   Slice sz : {SLICE_SIZE}

⚙️  MODEL
   Architecture : U-Net with skip connections
   Parameters   : {trainable_params:,}
   Loss         : MSE + SSIM (w={SSIM_WEIGHT})
   Optimizer    : Adam (lr={LEARNING_RATE})
   Epochs ran   : {len(history['train_loss'])}
   SR Factor    : {DOWNSAMPLE_FACTOR}x

📊 TEST SET PERFORMANCE
   Method                       PSNR (dB)    SSIM
   ─────────────────────────────────────────────
   Aliased LR Input             {np.mean(ablation['Aliased LR Input']['psnr']):.4f}       {np.mean(ablation['Aliased LR Input']['ssim']):.4f}
   Bicubic Interpolation        {np.mean(ablation['Bicubic Interpolation']['psnr']):.4f}       {np.mean(ablation['Bicubic Interpolation']['ssim']):.4f}
   U-Net (Our Model)            {test_psnr:.4f}       {test_ssim:.4f}
   ─────────────────────────────────────────────
   Improvement over Bicubic  {test_psnr - np.mean(ablation['Bicubic Interpolation']['psnr']):+.4f} dB    {test_ssim - np.mean(ablation['Bicubic Interpolation']['ssim']):+.4f}

💾 SAVED FILES
   best_smore_model.pth      ← Best model weights
   real_mri_slices.png       ← Dataset visualization
   degradation_simulation.png← LR/Aliasing simulation
   training_curves.png       ← Loss/PSNR/SSIM curves
   visual_comparison.png     ← Input vs GT vs Model
   ablation_study.png        ← Bar chart comparison
   edge_sharpness.png        ← Sobel edge analysis
""")

print("#" * 60)
print("✅ Project complete! All figures saved.")


---
## 💡 Tips to Improve Results

If your PSNR/SSIM is lower than expected, try these improvements:

| Action | Expected Effect |
|---|---|
| Increase `NUM_SUBJECTS` from 30 to 80+ | More training data → better generalization |
| Increase `NUM_EPOCHS` to 50–100 | More training → lower loss |
| Increase `base_features` from 32 to 64 | Larger model capacity |
| Tune `SSIM_WEIGHT` (default 0.2) | Balance PSNR vs structural quality |
| Reduce `DOWNSAMPLE_FACTOR` from 4 to 2 | Easier SR task → higher metrics |
| Add data augmentation (flips, rotations) | Better generalization |

---
## 📚 References

1. Zhao et al. (2021). *SMORE: A Self-supervised Anti-aliasing and Super-resolution Algorithm for MRI Using Deep Learning.* IEEE TMI. https://doi.org/10.1109/TMI.2020.3037187
2. IXI Dataset. https://brain-development.org/ixi-dataset/
3. Kaggle Dataset (CAT12/SPM preprocessed IXI). https://www.kaggle.com/datasets/hamedamin/preprocessed-oasis-and-epilepsy-and-ixi
4. Ronneberger et al. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation.* MICCAI.
